In [86]:
import torch.utils.data as data
import numpy as np
import os
import sys
from PIL import Image
import torch
import random
import math
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import subprocess
from collections import defaultdict

from torch.autograd import Variable
from torchinfo import summary

print(torchvision.__version__)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.current_device())

data_root_dir = "/scratch1/zeyut/eat_detection/"

0.9.1+cu102
1.8.1+cu102
True
0


In [104]:
class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
        
class GestureCountMatrix():
    def __init__(self,result_loc,gt_loc, downsample_rate, subject_wise=True):
        self.result_loc = result_loc
        self.pred_ges_loc = result_loc + "ges_boundaries/"
        self.pred_loc = result_loc + "frame_preds/"
        self.gt_loc = gt_loc
        self.downsample_rate = downsample_rate
        self.subject_wise = subject_wise
        self.video_list = [f[6:-4] for f in os.listdir(self.pred_loc) if f.startswith("preds")]

        self.subject_list = np.array([str.split(video,"_")[0] for video in self.video_list])
        self.subject_set = sorted(np.unique(self.subject_list),reverse=False)
        
    def CalculateGestureCount(self,write=False,smooth=True,smooth_kernel_params=[1,1,1,1]):
        self.CalculateBoundaries(write=write,smooth=smooth,smooth_kernel_params=smooth_kernel_params)
        
        matrixs = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: 0)))
        for video_idx in self.video_list:
            subject = video_idx
            if self.subject_wise:
                subject = str.split(video_idx,"_")[0]
            f_gt = open(f"{self.gt_loc}{video_idx}/gt_ges_3labels.txt","r")
            gt = []
            bite_average = AverageMeter()
            drink_average = AverageMeter()

            for line in f_gt.readlines():
                info = str.split(str.split(line, "\n")[0], "\t") 
                gt.append([LABEL_TABLE[info[0]],int(info[1]),int(info[2])])
                if info[0] == "bite":
                    bite_average.update(int(info[2])-int(info[1]))
                if info[0] == "drink":
                    drink_average.update(int(info[2])-int(info[1]))
            matrixs[subject][0]["average_len"] = bite_average.avg
            matrixs[subject][1]["average_len"] = drink_average.avg

            self.evaluate_results(gt, self.predictions[video_idx], matrixs[subject])

        summary_matrixs = defaultdict(lambda: defaultdict(lambda: 0))
        if self.subject_wise:
            loop_list = self.subject_set
        else:
            loop_list = self.video_list
        for ges_type in range(2):
            for key in ['tp','fp','missed','confused']:
                for subject in loop_list:
                    summary_matrixs[ges_type][key] += matrixs[subject][ges_type][key]                           
        self.summary_matrixs = summary_matrixs
        self.matrixs = matrixs

    def CalculateBoundaries(self, write=False,smooth=True,smooth_kernel_params=[1,1,1,1]):
        if write:
            try:
                os.mkdir(f"{result_loc}ges_boundaries/")
            except:
                pass
        self.predictions = {}
        for video_idx in self.video_list:
            preds, labels, start_idx = self.read_results(video_idx)
            if smooth:
                processed_preds = self.smooth_results(preds,
                                                      kernel1=smooth_kernel_params[0],
                                                      kernel2=smooth_kernel_params[1],
                                                      kernel3=smooth_kernel_params[2],
                                                      kernel4=smooth_kernel_params[3])
            else:
                processed_preds = preds
            cur_start = -1
            pred_boundaries = []
            pred_types = []
            pre_pred = -1
            for idx, pred in enumerate(processed_preds):
                if pred == pre_pred:
                    continue
                if cur_start == -1 and pred == 2:
                    continue
                if idx != 0 and cur_start != -1:
                    pred_types.append(pre_pred)
                    pred_boundaries.append([cur_start+start_idx,(idx-1)*self.downsample_rate+start_idx])
                    cur_start = -1
                else:
                    cur_start = idx * self.downsample_rate
                pre_pred = pred
            if write:
                f = open(f"{result_loc}ges_boundaries/ges_{video_idx}.txt","w")
                f.write("\n".join(["{}\t{}\t{}".\
                        format(ges_type,boundary[0],boundary[1]) \
                        for ges_type,boundary in (zip(pred_types,pred_boundaries))]))
                f.close()
            self.predictions[video_idx] = [[ges_type,boundary[0],boundary[1]] for ges_type,boundary in zip(pred_types,pred_boundaries)]
            
    def CalculateIndividualMatrix(self):
        average_precision = defaultdict(lambda: 0)
        average_recall = defaultdict(lambda: 0)
        bite_precision = defaultdict(lambda: 0) 
        bite_recall = defaultdict(lambda: 0) 
        
        if self.subject_wise:
            loop_list = self.subject_set
        else:
            loop_list = self.video_list
        for subject in loop_list:
            try:
                self.matrixs[subject][0]['precision'] = self.matrixs[subject][0]['tp']/(self.matrixs[subject][0]['tp']+self.matrixs[subject][0]['fp'])
            except:
                pass
            try:
                self.matrixs[subject][1]['precision'] = self.matrixs[subject][1]['tp']/(self.matrixs[subject][1]['tp']+self.matrixs[subject][1]['fp'])
            except:
                pass
            bite_precision[subject] = self.matrixs[subject][0]['precision']   
            # if there is a precision in "drink"
            if self.matrixs[subject][1]['tp']+self.matrixs[subject][1]['fp'] != 0:
                #average_precision[subject] = matrixs[subject][0]['precision']
                average_precision[subject] = (self.matrixs[subject][0]['precision'] + self.matrixs[subject][1]['precision']) / 2
            else:
                average_precision[subject] = self.matrixs[subject][0]['precision']

            try:
                self.matrixs[subject][0]['recall'] = self.matrixs[subject][0]['tp']/(self.matrixs[subject][0]['tp']+self.matrixs[subject][0]['missed'])
            except:
                pass
            try:
                self.matrixs[subject][1]['recall'] = self.matrixs[subject][1]['tp']/(self.matrixs[subject][1]['tp']+self.matrixs[subject][1]['missed'])
            except:
                pass
            try:
                self.matrixs[subject][0]['f1'] = self.matrixs[subject][0]['tp']/(self.matrixs[subject][0]['tp']+0.5*(self.matrixs[subject][0]['fp']+self.matrixs[subject][0]['missed']))
            except:
                pass
            try:
                self.matrixs[subject][1]['f1'] = self.matrixs[subject][1]['tp']/(self.matrixs[subject][1]['tp']+0.5*(self.matrixs[subject][1]['fp']+self.matrixs[subject][1]['missed']))
            except:
                pass
            bite_recall[subject] = self.matrixs[subject][0]['recall']   
            # if there is a recall in "drink"
            if self.matrixs[subject][1]['tp']+self.matrixs[subject][1]['missed'] != 0:
                average_recall[subject] = (self.matrixs[subject][0]['recall'] + self.matrixs[subject][1]['recall']) / 2
            else:
                average_recall[subject] = self.matrixs[subject][0]['recall']

        self.average_precision = average_precision
        self.average_recall = average_recall
        self.bite_precision = bite_precision
        self.bite_recall = bite_recall
        
        self.sorted_average_precision = {k: v for k, v in sorted(average_precision.items(), key=lambda item: item[1])}
        self.sorted_average_recall = {k: v for k, v in sorted(average_recall.items(), key=lambda item: item[1])}
        self.sorted_bite_precision = {k: v for k, v in sorted(bite_precision.items(), key=lambda item: item[1])}
        self.sorted_bite_recall = {k: v for k, v in sorted(bite_recall.items(), key=lambda item: item[1])}

    def PrintGlobalMatrix(self):
        print(f"\t\tbite\tdrink\t")
        print(f"=====summary=======================")
        print(f"TP\t\t{self.summary_matrixs[0]['tp']}\t{self.summary_matrixs[1]['tp']}")
        print(f"FP\t\t{self.summary_matrixs[0]['fp']}\t{self.summary_matrixs[1]['fp']}")
        print(f"Missed\t\t{self.summary_matrixs[0]['missed']}\t{self.summary_matrixs[1]['missed']}")
        print(f"Confused\t{self.summary_matrixs[0]['confused']}\t{self.summary_matrixs[1]['confused']}")
        print(f"Precision\t"
                f"{self.summary_matrixs[0]['tp']/(self.summary_matrixs[0]['tp']+self.summary_matrixs[0]['fp']):.4f}\t"
                f"{self.summary_matrixs[1]['tp']/(self.summary_matrixs[1]['tp']+self.summary_matrixs[1]['fp']):.4f}")
        print(f"Recall\t\t"
              f"{self.summary_matrixs[0]['tp']/(self.summary_matrixs[0]['tp']+self.summary_matrixs[0]['missed']):.4f}\t"
                f"{self.summary_matrixs[1]['tp']/(self.summary_matrixs[1]['tp']+self.summary_matrixs[1]['missed']):.4f}")
        print(f"F1 score\t"
              f"{self.summary_matrixs[0]['tp']/(self.summary_matrixs[0]['tp']+0.5*(self.summary_matrixs[0]['fp']+self.summary_matrixs[0]['missed'])):.4f}\t"
                f"{self.summary_matrixs[1]['tp']/(self.summary_matrixs[1]['tp']+0.5*(self.summary_matrixs[1]['fp']+self.summary_matrixs[1]['missed'])):.4f}")

    def PrintWorstIndividualMatrix(self, pivot=10):
        print(f"=====lowest {pivot} average precision=====")
        count = 0
        bad_video_list = []
        for k, v in self.sorted_bite_precision.items():
            print(f"{k}: {v:.2f}")
            bad_video_list.append(k)
            count += 1
            if count >= pivot:
                break
        print(f"=====lowest {pivot} average recall========")
        count = 0
        for k, v in self.sorted_bite_recall.items():
            print(f"{k}: {v:.2f}")
            bad_video_list.append(k)
            count += 1
            if count >= pivot:
                break
        bad_video_list = sorted(list(set(bad_video_list)))
        for subject in bad_video_list:
            print(f"====={subject}=========================")
            print(f"TP\t\t{self.matrixs[subject][0]['tp']}\t{self.matrixs[subject][1]['tp']}")
            print(f"FP\t\t{self.matrixs[subject][0]['fp']}\t{self.matrixs[subject][1]['fp']}")
            print(f"Missed\t\t{self.matrixs[subject][0]['missed']}\t{self.matrixs[subject][1]['missed']}")
            print(f"Confused\t{self.matrixs[subject][0]['confused']}\t{self.matrixs[subject][1]['confused']}")
            print(f"Precision\t"
                  f"{self.matrixs[subject][0]['precision']:.2f}\t"
                  f"{self.matrixs[subject][1]['precision']:.2f}")
            print(f"Recall\t\t"
                  f"{self.matrixs[subject][0]['recall']:.2f}\t"
                f"{self.matrixs[subject][1]['recall']:.2f}")
            print(f"average len\t"
                  f"{self.matrixs[subject][0]['average_len']:.2f}\t"
                f"{self.matrixs[subject][1]['average_len']:.2f}")

    def PrintSingleMatrix(self, subject):
        print(f"====={subject}=========================")
        print(f"TP\t\t{self.matrixs[subject][0]['tp']}\t{self.matrixs[subject][1]['tp']}")
        print(f"FP\t\t{self.matrixs[subject][0]['fp']}\t{self.matrixs[subject][1]['fp']}")
        print(f"Missed\t\t{self.matrixs[subject][0]['missed']}\t{self.matrixs[subject][1]['missed']}")
        print(f"Confused\t{self.matrixs[subject][0]['confused']}\t{self.matrixs[subject][1]['confused']}")
        print(f"Precision\t"
            f"{self.matrixs[subject][0]['precision']:.2f}\t"
            f"{self.matrixs[subject][1]['precision']:.2f}")
        print(f"Recall\t\t"
            f"{self.matrixs[subject][0]['recall']:.2f}\t"
            f"{self.matrixs[subject][1]['recall']:.2f}")
        print(f"average len\t"
            f"{self.matrixs[subject][0]['average_len']:.2f}\t"
            f"{self.matrixs[subject][1]['average_len']:.2f}")
    
    
    def smooth_results(self, results, kernel1=1,kernel2=1,kernel3=1,kernel4=1):
        #results,kernel1=4,kernel2=2,kernel3=3,kernel4=6
        #results,kernel1=8,kernel2=4,kernel3=8,kernel4=16
        """
        kernel1: for filling small gaps between bite gestures
        kernel2: for filling small gaps between drink gestures
        kernel3: for removing short bite gestures
        kernel4: for removing short drink gestures
        """
        smoothed = np.copy(results)
        for i in range(kernel1,len(results)):
            if results[i-kernel1+1] == results[i] and results[i] == 0:
                smoothed[i-kernel1+2:i] = results[i-kernel1+1]
        results = smoothed
        for i in range(kernel2,len(results)):
            if results[i-kernel2+1] == results[i] and results[i] == 1:
                smoothed[i-kernel2+2:i] = results[i-kernel2+1]
        results = smoothed
        for i in range(kernel3,len(results)):
            if results[i-kernel3+1] == results[i] and \
                results[i] == 2 and \
                0 in results[i-kernel3+2:i] and \
                1 not in results[i-kernel4+2:i]:
                smoothed[i-kernel3+2:i] = 2
        results = smoothed
        for i in range(kernel4,len(results)):
            if results[i-kernel4+1] == results[i] and \
                results[i] == 2 and \
                1 in results[i-kernel4+2:i] and \
                0 not in results[i-kernel4+2:i]:
                smoothed[i-kernel4+2:i] = 2
        return smoothed
    
    
    def read_results(self,video_idx):
        min_num = 0
        preds = []
        labels = [] 
        results = open("{}preds_{}.txt".format(self.pred_loc,video_idx),"r")
        for line in results.readlines():
            image_name = line.split("\t")[0]
            preds.append(int(line.split("\t")[2].split("\n")[0]))
            labels.append(int(line.split("\t")[1].split("\n")[0]))

            if not min_num:
                min_num = int(image_name[6:12])
        return np.array(preds),np.array(labels), min_num

    def evaluate_results(self, gt, pred, matrixs):
        #handle edge cases when gt or pred is empty
        if len(gt) == 0 and len(pred) == 0:
            return
        elif len(gt) == 0:
            matrixs[pred[0][0]]["fp"] += 1
            return self.evaluate_results(gt, pred[1:], matrixs)
        elif len(pred) == 0:
            matrixs[gt[0][0]]["missed"] += 1
            return self.evaluate_results(gt[1:], pred, matrixs)
        else:
            # read in first gesture in gt and pred, respectively
            gt_start = gt[0][1]
            gt_end = gt[0][2]
            gt_type = gt[0][0]
            pred_start = pred[0][1]
            pred_end = pred[0][2]
            pred_type = pred[0][0]
            overlap = min(pred_end-gt_start,gt_end-pred_start)  
            # the gt gesture and predicted gesture are considered matched when
            # the overlap is above 50% of either of them.
            if overlap < (gt_end-gt_start)*0.5 and \
                    overlap < (pred_end-pred_start)*0.5:
                if pred_start < gt_start:
                    matrixs[pred_type]["fp"] += 1
                    return self.evaluate_results(gt, pred[1:], matrixs)
                else:
                    matrixs[gt_type]["missed"] += 1
                    return self.evaluate_results(gt[1:], pred, matrixs)  
            else:
                if gt_type == pred_type:
                    matrixs[gt_type]["tp"] += 1
                else:
                    matrixs[gt_type]["confused"] += 1
                return self.evaluate_results(gt[1:], pred[1:], matrixs)


In [105]:
LABEL_TABLE = {"bite": 0, "drink": 1, "non_intake": 2}
gt_loc = data_root_dir + "all_labels/"
model_idx = 2
first_model_result = GestureCountMatrix(result_loc = data_root_dir + f"results_10runs/{model_idx}/result_RES_LSTM_30_16_8_v4_test/",
                                      gt_loc = data_root_dir + "all_labels/",
                                      downsample_rate=1,
                                      subject_wise=False)
second_model_result = GestureCountMatrix(result_loc = data_root_dir + f"/result_single_lstm_50_16_8_v4_2stage_all/{model_idx}/",
                                       gt_loc = data_root_dir + "all_labels/",
                                      downsample_rate=4,
                                      subject_wise=False)


In [106]:
first_model_result.CalculateGestureCount(write=False,smooth=False)
second_model_result.CalculateGestureCount(write=False,smooth=False)
first_model_result.PrintGlobalMatrix()
second_model_result.PrintGlobalMatrix()

		bite	drink	
=====summary=======================
TP		2651	424
FP		1457	256
Missed		364	124
Confused	10	4
Precision	0.6453	0.6235
Recall		0.8793	0.7737
F1 score	0.7443	0.6906
		bite	drink	
=====summary=======================
TP		2658	449
FP		460	48
Missed		362	100
Confused	5	3
Precision	0.8525	0.9034
Recall		0.8801	0.8179
F1 score	0.8661	0.8585


### Frame-wise result evaluation

In [17]:
pred_frame_loc = result_loc + "frame_preds/"
video_list = [f[6:-4] for f in os.listdir(pred_frame_loc) if f.startswith("preds")]
frame_matrix = [defaultdict(lambda: defaultdict(lambda: 0)) for _ in range(3)]

total_tp = np.array([0,0,0])
total_p = np.array([0,0,0])
total_detect = np.array([0,0,0])
subject_list = np.array([str.split(video,"_")[0] for video in video_list])
subject_set = sorted(np.unique(subject_list),reverse=False)
#bad_subjects = ['p139','p285','p361']
bad_subjects = []
for video_idx in video_list:
    subject = str.split(video_idx,"_")[0]
    frame_gt = []
    f_pred = open(f"{pred_frame_loc}preds_{video_idx}.txt","r")
    frame_preds = []
    for line in f_pred.readlines():
        info = str.split(str.split(line, "\n")[0], "\t") 
        frame_gt.append(int(info[1]))
        frame_preds.append(int(info[2]))
    frame_preds = np.array(frame_preds)
    frame_gt = np.array(frame_gt)
    f_pred.close()
    cur_uar = 0

    for label in range(3):
        cur_tp = np.logical_and(frame_preds == frame_gt, frame_gt==label).sum()
        cur_p = (frame_gt == label).sum().item()
        cur_detect = (frame_preds == label).sum().item()
        frame_matrix[label][subject]["tp"] += cur_tp
        frame_matrix[label][subject]["p"] += cur_p
        frame_matrix[label][subject]["detect"] += cur_detect
        total_tp[label] += cur_tp
        total_p[label] += cur_p
        total_detect[label] += cur_detect

In [20]:
recall = total_tp/total_p
precision = total_tp/total_detect
f1 = 2 * (recall*precision) / (recall+precision)
frame_uar = np.mean(total_tp/total_p)

print(f"for all frames")
print(f"average uar: {frame_uar}")
print(f"tp: {total_tp}")
print(f"p: {total_p}")
print(f"detection: {total_detect}")
print(f"precision: {precision}")
print(f"recall: {recall}")
print(f"f1 {f1}")

for all frames
average uar: 0.8121212194304416
tp: [ 36430  21779 321511]
p: [ 56508  24876 350928]
detection: [ 56600  31595 344117]
precision: [0.64363958 0.68931793 0.93430723]
recall: [0.64468748 0.87550249 0.91617369]
f1 [0.6441631  0.77133396 0.92515161]


In [57]:
uar_dict = defaultdict(lambda: 0) 
recall_dict = [defaultdict(lambda: 0) for _ in range(3)]
precision_dict = [defaultdict(lambda: 0) for _ in range(3)]
f1_dict = [defaultdict(lambda: 0) for _ in range(3)]
for subject in subject_set:
    for label in range(3):
        recall_dict[label][subject] = frame_matrix[label][subject]["tp"] / frame_matrix[label][subject]["p"]
        precision_dict[label][subject] = frame_matrix[label][subject]["tp"] / frame_matrix[label][subject]["detect"]
        f1_dict[label][subject] = 2 * (recall_dict[label][subject]*precision_dict[label][subject]) / (recall_dict[label][subject]+precision_dict[label][subject])
        uar_dict[subject] += recall_dict[label][subject]
    uar_dict[subject] = uar_dict[subject]/3    
    
recall_list = []
precision_list = []
f1_list = []
uar_list = list(uar_dict.values())
for label in range(3):
    recall_list.append(list(recall_dict[label].values()))
    precision_list.append(list(precision_dict[label].values()))
    f1_list.append(list(f1_dict[label].values()))

In [59]:
print(f"uar: mean: {np.mean(uar_list)}; std: {np.std(uar_list)}")

uar: mean: 0.8120667934254321; std: 0.05222157203610562


In [54]:
print(f"for participants")
print(f"recall for bite: mean: {np.mean(recall_list[0])}; std: {np.std(recall_list[0])}")
print(f"recall for drink: mean: {np.mean(recall_list[1])}; std: {np.std(recall_list[1])}")
print(f"recall for non-intake: mean: {np.mean(recall_list[2])}; std: {np.std(recall_list[2])}")
print("------------------------------------------------------------")
print(f"precision for bite: mean: {np.mean(precision_list[0])}; std: {np.std(precision_list[0])}")
print(f"precision for drink: mean: {np.mean(precision_list[1])}; std: {np.std(precision_list[1])}")
print(f"precision for non-intake: mean: {np.mean(precision_list[2])}; std: {np.std(precision_list[2])}")
print("------------------------------------------------------------")
print(f"f1 for bite: mean: {np.mean(f1_list[0])}; std: {np.std(f1_list[0])}")
print(f"f1 for drink: mean: {np.mean(f1_list[1])}; std: {np.std(f1_list[1])}")
print(f"f1 for non-intake: mean: {np.mean(f1_list[2])}; std: {np.std(f1_list[2])}")


for participants
recall for bite: mean: 0.6517256544030687; std: 0.12765630732780464
recall for drink: mean: 0.8692000454393204; std: 0.09016816242604321
recall for non-intake: mean: 0.9152746804339073; std: 0.05285680338252555
------------------------------------------------------------
precision for bite: mean: 0.6957560583364025; std: 0.1557008473022539
precision for drink: mean: 0.70500832099051; std: 0.17861026950221467
precision for non-intake: mean: 0.9322267013355503; std: 0.03218550825745765
------------------------------------------------------------
f1 for bite: mean: 0.6484365172376624; std: 0.073171626550479
f1 for drink: mean: 0.766571719232751; std: 0.14693386253383228
f1 for non-intake: mean: 0.922667662508028; std: 0.03286377776620898
